## 00. Quick Start


In [1]:
print('Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.')
print('LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.')

Concept Portfolio V2 Lab — staged와 one-click은 동일 Core를 사용합니다.
LIVE_TEST_LEVEL로 CORE / LEGAL_C1 / FULL_E2E / ONE_CLICK 중 하나만 선택하세요.


## 01. Environment


In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display
SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
AI_ROOT = next((p for p in SEARCH_ROOTS if (p / 'app').is_dir()), None)
if AI_ROOT is None: AI_ROOT = next((p / 'ai' for p in SEARCH_ROOTS if (p / 'ai' / 'app').is_dir()))
if str(AI_ROOT) not in sys.path: sys.path.insert(0, str(AI_ROOT))
print({'python': sys.version.split()[0], 'aiRoot': str(AI_ROOT)})

{'python': '3.14.5', 'aiRoot': 'C:\\Users\\seewo\\Desktop\\big_proj_01\\new_3\\ai'}


## 02. MODE


In [3]:
MODE = 'LIVE'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
print({'mode': MODE, 'liveExternalOperationsEnabled': MODE == 'LIVE'})

{'mode': 'LIVE', 'liveExternalOperationsEnabled': True}


## 03. Environment Check


In [4]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
env_status = {key: bool(os.getenv(key)) for key in LIVE_ENV_KEYS}
print(env_status if MODE == 'LIVE' else {'mode': MODE, 'message': '외부 환경변수 불필요'})

{'AI_PROVIDER': True, 'AI_API_KEY': True, 'AI_MODEL': True, 'MOLEG_API_KEY': True, 'LEGAL_REGISTRY_VERSION': True}


## 04. Schema Preflight


In [5]:
from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.adapters import CurrentLegalAdapter
from app.concept_portfolio_v2.diagnostics.notebook_view import *
gateway = ProviderGateway(MODE, recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(MODE, gateway=gateway)
schema_preflight = engine.schema_preflight_report()
display(show_schema_preflight(schema_preflight))
assert schema_preflight.status == 'PASS' and schema_preflight.providerCalls == 0

,스키마,상태,실패,Provider 호출
0,PlanDraftPool,PASS,[],0
1,ConceptCandidateDraft,PASS,[],0
2,SemanticDistinctnessResult,PASS,[],0
3,SemanticFidelityResult,PASS,[],0
4,SemanticArchitectureBatch,PASS,[],0
5,SemanticHypothesisBatch,PASS,[],0
6,BusinessRoleSemanticBatch,PASS,[],0
7,LegalFactDependencySemanticBatch,PASS,[],0
8,LegalFactCompletionPatch,PASS,[],0


## 05. Input


In [6]:
SCENARIO_FILE = AI_ROOT / 'fixtures' / 'concept_portfolio_v2' / 'live_scenarios.json'
SCENARIOS = {item['scenarioId']: item for item in json.loads(SCENARIO_FILE.read_text(encoding='utf-8'))}
LIVE_SCENARIO = 'WEEKEND_TRIP_PLANNER'
LIVE_TEST_LEVEL = 'FULL_E2E'  # CORE | LEGAL_C1 | FULL_E2E | ONE_CLICK
RUN_STAGED_CORE = LIVE_TEST_LEVEL in {'CORE', 'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_LEGAL = LIVE_TEST_LEVEL in {'LEGAL_C1', 'FULL_E2E'}
RUN_STAGED_FULL = LIVE_TEST_LEVEL == 'FULL_E2E'
scenario = SCENARIOS[LIVE_SCENARIO]
TEST_INPUT = {key: scenario[key] for key in ('ideaOverview', 'problem', 'targetUsers')}
MAX_CONCEPTS = 5
display({'scenario': LIVE_SCENARIO, 'testLevel': LIVE_TEST_LEVEL, 'domain': scenario['domain'],
         'expectedStructuralFeatures': scenario['expectedStructuralFeatures']})

{'scenario': 'WEEKEND_TRIP_PLANNER',
 'testLevel': 'FULL_E2E',
 'domain': 'Travel planning',
 'expectedStructuralFeatures': ['디지털 계획', '예약 연결 역할과 직접 판매 구분']}

## 06. Idea Brief Derivation


In [7]:
seed = engine.seed_adapter.adapt(TEST_INPUT)
idea_context = None
if RUN_STAGED_CORE:
    engine._reset()
    idea_context = await engine.derive_idea_brief(seed)
print({'ideaCallComplete': bool(idea_context), 'interpretationPresent': bool(seed.interpretation),
       'stagedCore': RUN_STAGED_CORE})

{'ideaCallComplete': True, 'interpretationPresent': True, 'stagedCore': True}


## 07. Safety


In [8]:
display(idea_context.safetyReview.model_dump(mode='json') if idea_context else {'status': 'SKIPPED'})
assert idea_context is None or idea_context.safetyReview.passed

{'decision': 'ALLOW',
 'categories': [],
 'restrictions': [],
 'userFacingReason': '이 아이디어는 안전하며, 여행 계획 서비스를 제공하는 것으로 보입니다.'}

## 08. AI가 이해한 아이디어


In [9]:
display(show_idea_interpretation(idea_context) if idea_context else {'status': 'SKIPPED'})

,항목,AI 이해 결과
0,interpretedProblem,짧은 주말 여행을 계획하는 데 소요되는 시간과 노력을 줄이는 것.
1,interpretedTargetUsers,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.
2,usageContext,사용자의 이동 가능 시간과 관심사를 고려하여 여행 일정을 자동으로 구성하는 서비스.
3,industryCategory,여행 및 관광
4,researchScope,주말 여행 계획 서비스에 대한 시장 조사.
5,conciseIdeaDefinition,사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동으로 구성...
6,targetRegionInterpretation,대상 지역은 명시되지 않음.
7,relevantKnownCompetitorContext,경쟁자는 명시되지 않음.


## 09. Readiness / Summary / commitments


In [10]:
display(show_idea_readiness(idea_context) if idea_context else {'status': 'SKIPPED'})

{'readiness': {'status': 'READY_FOR_REVIEW',
  'score': 0,
  'missingFieldKeys': []},
 'readinessDiagnostic': 'READINESS_INCONSISTENT',
 'userFacingSummary': '이 아이디어는 사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동으로 구성하는 여행 계획 서비스입니다.',
 'commitmentCandidates': [],
 'contradictions': [],
 'questions': []}

## 10. Seed Analysis


In [11]:
analysis = await engine.analyze_seed(seed) if RUN_STAGED_CORE else None
display(show_seed_analysis(analysis) if analysis else {'status': 'SKIPPED'})

,구분,값
0,탐색 폭,EXPLORE
1,다양성 수용량,5
2,설명,선택 입력 LOCK 0개로 11개 설계 차원이 열려 있습니다. diversityCa...


## 11. Generic Opportunity Kernel


In [12]:
display(analysis.opportunityKernel.model_dump(mode='json') if analysis else {'status': 'SKIPPED'})

{'problemCore': '짧은 주말 여행을 계획하는 데 소요되는 시간과 노력을 줄이는 것.',
 'targetCore': '주말에 1~2일 단기 여행을 자주 계획하는 직장인들.',
 'useContexts': ['사용자의 이동 가능 시간과 관심사를 고려하여 여행 일정을 자동으로 구성하는 서비스.'],
 'intentComponents': ['사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동으로 구성하는 서비스.'],
 'mustPreserve': ['짧은 주말 여행을 계획하는 데 소요되는 시간과 노력을 줄이는 것.',
  '주말에 1~2일 단기 여행을 자주 계획하는 직장인들.',
  '사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동으로 구성하는 서비스.'],
 'maySpecialize': ['핵심 대상의 의미 있는 하위 세그먼트',
  '핵심 사용 맥락의 구체화',
  '가치 제안 또는 offer의 구체화'],
 'forbiddenDriftSummary': '핵심 문제와 대상이 모두 무관한 기회로 교체되면 범위를 벗어납니다.'}

## 12. Design Space


In [13]:
display(show_design_space(analysis) if analysis else {'status': 'SKIPPED'})

,분류,필드,값
0,SOURCE_LOCK,ideaOverview,사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동 구성하고...
1,SOURCE_LOCK,problem,짧은 주말 여행은 이동 시간까지 고려해 일정을 직접 조합하는 데 많은 시간이 든다.
2,SOURCE_LOCK,targetUsers,주말 1~2일 단기 여행을 자주 계획하는 직장인
3,SEMANTIC_ANCHOR,ideaOverview,사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동 구성하고...
4,SEMANTIC_ANCHOR,problem,짧은 주말 여행은 이동 시간까지 고려해 일정을 직접 조합하는 데 많은 시간이 든다.
5,SEMANTIC_ANCHOR,targetUsers,주말 1~2일 단기 여행을 자주 계획하는 직장인
6,OPEN,solutionMechanism,변경 가능
7,OPEN,valueDelivery,변경 가능
8,OPEN,operatingModel,변경 가능
9,OPEN,supplyStructure,변경 가능


## 13. Generate and Adaptively Replenish Plan Pool


In [14]:
plan_validation = (await engine.prepare_portfolio_plans(seed, analysis, max_concepts=MAX_CONCEPTS)
                   if RUN_STAGED_CORE else None)
plans = engine._last_plan_pool if plan_validation else []
print({'totalPlans': len(plans),
       'planningRounds': plan_validation.planningRounds if plan_validation else 0,
       'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0})

{'totalPlans': 6, 'planningRounds': 1, 'replenishmentRequested': 0}


## 14. Plan Count / Adaptive Replenishment Check


In [15]:
display(show_plan_pool_status(engine._last_plan_pool_status) if plan_validation else {'status': 'SKIPPED'})
display({'planningRounds': plan_validation.planningRounds if plan_validation else 0,
         'replenishmentRequested': plan_validation.replenishmentRequested if plan_validation else 0,
         'adaptiveReplenishmentUsed': bool(plan_validation and plan_validation.planningRounds > 1)})

,requestedPoolSize,returnedPoolSize,initialTarget,reserveTarget,reserveAvailable,status
0,7,6,5,2,1,RESERVE_SHORTFALL


{'planningRounds': 1,
 'replenishmentRequested': 0,
 'adaptiveReplenishmentUsed': False}

## 15. Korean Plan Display


In [16]:
display(show_portfolio_plans(plan_validation.acceptedPlans + plan_validation.reservePlans)
        if plan_validation else {'status': 'SKIPPED'})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,주말 여행 자동 계획 서비스,SELECTED,0.9035,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 이동 가능 시간과 관심사를 고려하여 여행 일정을 자동으로 구성하는 서비스.,"사용자는 여행 계획에 소요되는 시간을 절약하고, 더 많은 시간을 여행에 할애할 수 있다.",사용자의 이동 가능 시간과 관심사를 기반으로 최적의 여행 일정을 자동으로 생성하여 ...,자동으로 생성된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...","주말 여행 계획의 효율성을 높이고, 사용자 경험을 개선하기 위해."
1,P4,주말 여행 계획 지원 서비스,SELECTED,0.8399,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 이동 가능 시간과 관심사를 고려하여 여행 계획을 지원하는 서비스.,사용자는 여행 계획을 쉽게 진행할 수 있다.,여행 계획을 지원하여 사용자의 편의를 높인다.,자동으로 생성된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공한다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",여행 계획의 편리함을 제공하여 사용자 경험을 개선하기 위해.
2,P3,주말 여행 일정 최적화 서비스,SELECTED,0.6773,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 이동 가능 시간과 관심사를 고려하여 최적화된 여행 일정을 제공하는 서비스.,"사용자는 여행 계획에 소요되는 시간을 줄이고, 비용을 절감할 수 있다.",최적화된 여행 일정을 제공하여 사용자의 시간과 비용을 절약한다.,자동으로 생성된 최적화된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공한다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...","여행 일정의 효율성을 높이고, 사용자 경험을 개선하기 위해."
3,P5,주말 여행 일정 생성 서비스,SELECTED,0.6711,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 요구와 이동 가능 시간을 고려하여 여행 일정을 생성하는 서비스.,사용자는 자신의 요구에 맞는 여행 일정을 쉽게 찾을 수 있다.,사용자의 요구를 분석하여 최적의 여행 일정을 생성한다.,생성된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",사용자의 요구를 반영한 여행 일정 생성을 통해 만족도를 높이기 위해.
4,P6,주말 여행 일정 추천 및 예약 서비스,SELECTED,0.5646,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,추천된 여행 일정을 사용자가 쉽게 예약할 수 있도록 지원하는 서비스.,사용자는 추천된 일정을 쉽게 예약할 수 있다.,추천된 여행 일정을 사용자가 쉽게 예약할 수 있도록 지원한다.,추천된 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",추천된 일정을 쉽게 예약할 수 있도록 하여 사용자 경험을 개선하기 위해.
5,P2,주말 여행 맞춤형 추천 서비스,RESERVE,0.5775,Selected Portfolio 대비 marginal value가 낮아 reser...,"AMBIGUOUS,DISTINCT,VARIANT",거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 취향과 이동 가능 시간을 고려하여 맞춤형 여행 일정을 추천하는 서비스.,사용자는 자신의 취향에 맞는 여행 일정을 쉽게 찾을 수 있다.,사용자의 취향을 분석하여 최적의 여행 일정을 추천한다.,추천된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",사용자의 취향을 반영한 여행 추천으로 만족도를 높이기 위해.


## 16. Plan Lock/Intent Validation


In [17]:
display({'accepted': [p.planId for p in plan_validation.acceptedPlans] if plan_validation else [],
         'rejected': [p.model_dump(mode='json') for p in plan_validation.rejectedPlans]
                     if plan_validation else []})

{'accepted': ['P1', 'P4', 'P3', 'P5', 'P6'], 'rejected': []}

## 17. Portfolio Family / Variant / Distinct


In [18]:
display(show_plan_diversity(plan_validation.diversity) if plan_validation else {'status': 'SKIPPED'})

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,P1,P2,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
1,P1,P3,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,P2,P3,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, useCaseThesis, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
3,P1,P4,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, valuePropositionThesis, offerThe...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
4,P2,P4,DISTINCT,INTERMEDIARY:OTHER,OTHER:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, useCaseThesis, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
5,P3,P4,VARIANT,OTHER:OTHER,OTHER:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
6,P1,P5,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","useCaseThesis, valuePropositionThesis, offerTh...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
7,P2,P5,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole: INTERMEDIARY, customerInteractio...",mechanismFamily: 추천된 여행 일정은 사용자가 선호하는 숙박 및 체험 ...,SEMANTIC_REVIEW,True,"두 개의 아키텍처는 비즈니스 역할, 고객 상호작용 모델, 데이터 의존성, 배달 모델..."
8,P3,P5,DISTINCT,OTHER:OTHER,INTERMEDIARY:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, useCaseThesis, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,P4,P5,DISTINCT,OTHER:OTHER,INTERMEDIARY:OTHER,"operatingModel, partnerModel, deliveryModel, t...","businessRole, useCaseThesis, valuePropositionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.


## 18. Selected + Reserve Plans


In [19]:
selected_plans = plan_validation.acceptedPlans if plan_validation else []
reserve_plans = plan_validation.reservePlans if plan_validation else []
display(show_portfolio_plans(selected_plans + reserve_plans))
display({'selected': [p.planId for p in selected_plans], 'reserve': [p.planId for p in reserve_plans]})

,planId,제목,선택 상태,selectionScore,selectionReason,relationToPortfolio,Concept Family,Target Thesis,Use Context,Value Thesis,Offer Thesis,Solution Thesis,Architecture,비교 가치
0,P1,주말 여행 자동 계획 서비스,SELECTED,0.9035,Opportunity fit과 Concept clarity가 가장 높은 대표안,PORTFOLIO_SEED,거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 이동 가능 시간과 관심사를 고려하여 여행 일정을 자동으로 구성하는 서비스.,"사용자는 여행 계획에 소요되는 시간을 절약하고, 더 많은 시간을 여행에 할애할 수 있다.",사용자의 이동 가능 시간과 관심사를 기반으로 최적의 여행 일정을 자동으로 생성하여 ...,자동으로 생성된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...","주말 여행 계획의 효율성을 높이고, 사용자 경험을 개선하기 위해."
1,P4,주말 여행 계획 지원 서비스,SELECTED,0.8399,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 이동 가능 시간과 관심사를 고려하여 여행 계획을 지원하는 서비스.,사용자는 여행 계획을 쉽게 진행할 수 있다.,여행 계획을 지원하여 사용자의 편의를 높인다.,자동으로 생성된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공한다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...",여행 계획의 편리함을 제공하여 사용자 경험을 개선하기 위해.
2,P3,주말 여행 일정 최적화 서비스,SELECTED,0.6773,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,기타 역할 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 이동 가능 시간과 관심사를 고려하여 최적화된 여행 일정을 제공하는 서비스.,"사용자는 여행 계획에 소요되는 시간을 줄이고, 비용을 절감할 수 있다.",최적화된 여행 일정을 제공하여 사용자의 시간과 비용을 절약한다.,자동으로 생성된 최적화된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공한다.,"{'businessRole': 'OTHER', 'operatingModel': 'O...","여행 일정의 효율성을 높이고, 사용자 경험을 개선하기 위해."
3,P5,주말 여행 일정 생성 서비스,SELECTED,0.6711,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 요구와 이동 가능 시간을 고려하여 여행 일정을 생성하는 서비스.,사용자는 자신의 요구에 맞는 여행 일정을 쉽게 찾을 수 있다.,사용자의 요구를 분석하여 최적의 여행 일정을 생성한다.,생성된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",사용자의 요구를 반영한 여행 일정 생성을 통해 만족도를 높이기 위해.
4,P6,주말 여행 일정 추천 및 예약 서비스,SELECTED,0.5646,현재 Portfolio에 주요 사업 선택의 비교 범위를 추가,DISTINCT,거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,추천된 여행 일정을 사용자가 쉽게 예약할 수 있도록 지원하는 서비스.,사용자는 추천된 일정을 쉽게 예약할 수 있다.,추천된 여행 일정을 사용자가 쉽게 예약할 수 있도록 지원한다.,추천된 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",추천된 일정을 쉽게 예약할 수 있도록 하여 사용자 경험을 개선하기 위해.
5,P2,주말 여행 맞춤형 추천 서비스,RESERVE,0.5775,Selected Portfolio 대비 marginal value가 낮아 reser...,"AMBIGUOUS,DISTINCT,VARIANT",거래 중개 · 기타 운영,주말에 1~2일 단기 여행을 자주 계획하는 직장인들.,사용자의 취향과 이동 가능 시간을 고려하여 맞춤형 여행 일정을 추천하는 서비스.,사용자는 자신의 취향에 맞는 여행 일정을 쉽게 찾을 수 있다.,사용자의 취향을 분석하여 최적의 여행 일정을 추천한다.,추천된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,"{'businessRole': 'INTERMEDIARY', 'operatingMod...",사용자의 취향을 반영한 여행 추천으로 만족도를 높이기 위해.


{'selected': ['P1', 'P4', 'P3', 'P5', 'P6'], 'reserve': ['P2']}

## 19. Candidate 1


In [20]:
candidate_one = (await engine.expand_plan(seed, selected_plans[0], 1)
                 if RUN_STAGED_CORE and selected_plans else None)
display(show_candidates([candidate_one]) if candidate_one else [])

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,주말 여행 자동 계획 서비스,자동으로 생성된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,앱 내 예약 수수료를 통한 수익 창출,사용자 데이터를 기반으로 지속적으로 알고리즘을 개선하여 더 나은 여행 일정을 제공한다.


## 20. Candidate 1 Korean/Governance


In [21]:
candidate_one_reports = []
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'status': 'PENDING_FULL_CANDIDATE_RECOVERY'})

{'candidateId': 'C1', 'status': 'PENDING_FULL_CANDIDATE_RECOVERY'}

## 21. Candidate 1 Actual Generic Descriptor


In [22]:
display(show_concept_descriptors([candidate_one]) if candidate_one else [])

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,INTERMEDIARY:OTHER,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,INTERMEDIARY:OTHER,deliveryModel,OTHER,LOW,UNKNOWN
4,C1,INTERMEDIARY:OTHER,transactionModel,BOOKING,HIGH,RULE
5,C1,INTERMEDIARY:OTHER,monetizationModel,COMMISSION,HIGH,RULE
6,C1,INTERMEDIARY:OTHER,customerInteractionModel,APP,HIGH,RULE
7,C1,INTERMEDIARY:OTHER,dataDependency,MATERIAL,NaN,NaN
8,C1,INTERMEDIARY:OTHER,physicalDependency,MATERIAL,NaN,NaN


## 22. Candidate 1 Fidelity


In [23]:
display({'candidateId': candidate_one.candidateId if candidate_one else None,
         'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'})

{'candidateId': 'C1',
 'fidelity': '전체 Candidate Recovery 단계에서 semantic fallback 포함 검증'}

## 23. Remaining Candidates


In [24]:
remaining_candidates = []
if RUN_STAGED_CORE:
    for i, plan in enumerate(selected_plans[1:], 2):
        remaining_candidates.append(await engine.expand_plan(seed, plan, i))
candidate_drafts = ([candidate_one] if candidate_one else []) + remaining_candidates
display(show_candidates(candidate_drafts))

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C1,L1,None,주말 여행 자동 계획 서비스,자동으로 생성된 여행 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,앱 내 예약 수수료를 통한 수익 창출,사용자 데이터를 기반으로 지속적으로 알고리즘을 개선하여 더 나은 여행 일정을 제공한다.
1,C2,L2,None,주말 여행 계획 지원 서비스,자동으로 생성된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공한다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,"수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다.",사용자 데이터를 기반으로 알고리즘을 지속적으로 개선하여 더 나은 여행 계획을 지원한다.
2,C3,L3,None,주말 여행 일정 최적화 서비스,자동으로 생성된 최적화된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공합니다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,"수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출합니다.",사용자 데이터를 기반으로 알고리즘을 지속적으로 개선하여 최적의 일정을 제공합니다.
3,C4,L4,None,주말 여행 일정 생성 서비스,"사용자의 요구를 분석하는 알고리즘을 통해 여행 일정을 생성하고, 생성된 일정은 사용...",거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,"사용자는 생성된 일정에 따라 직접 예약을 진행하며, 수수료를 통해 수익을 창출한다.",사용자 피드백을 통해 생성 알고리즘을 지속적으로 개선한다.
4,C5,L5,None,주말 여행 일정 추천 및 예약 서비스,추천된 일정은 사용자가 선호하는 숙박 및 체험 예약 페이지로 연결된다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,예약 수수료 기반 수익 모델,사용자 데이터를 기반으로 추천 알고리즘을 지속적으로 개선하여 더 나은 일정을 제공한다.


## 24. Candidate Actual Generic Descriptors


In [25]:
display(show_concept_descriptors(candidate_drafts))

,entityId,family,dimension,code,confidence,source
0,C1,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE
1,C1,INTERMEDIARY:OTHER,operatingModel,OTHER,LOW,UNKNOWN
2,C1,INTERMEDIARY:OTHER,partnerModel,PARTNER_NETWORK,HIGH,RULE
3,C1,INTERMEDIARY:OTHER,deliveryModel,OTHER,LOW,UNKNOWN
4,C1,INTERMEDIARY:OTHER,transactionModel,BOOKING,HIGH,RULE
5,C1,INTERMEDIARY:OTHER,monetizationModel,COMMISSION,HIGH,RULE
6,C1,INTERMEDIARY:OTHER,customerInteractionModel,APP,HIGH,RULE
7,C1,INTERMEDIARY:OTHER,dataDependency,MATERIAL,NaN,NaN
8,C1,INTERMEDIARY:OTHER,physicalDependency,MATERIAL,NaN,NaN
9,C2,INTERMEDIARY:OTHER,businessRole,INTERMEDIARY,HIGH,RULE


## 25. Candidate Recovery / Portfolio Relations


In [26]:
candidate_preparation = (await engine.prepare_candidate_portfolio(
    seed, plan_validation, max_concepts=MAX_CONCEPTS, initial_candidates=candidate_drafts)
    if RUN_STAGED_CORE and plan_validation else None)
candidates = candidate_preparation.candidates if candidate_preparation else []
candidate_reports = candidate_preparation.reports if candidate_preparation else []
display(show_candidate_recovery(candidate_preparation) if candidate_preparation else {'status': 'SKIPPED'})
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
display(show_plan_diversity(candidate_pairwise))

{'summary':    candidateGenerated  candidateAcceptedInitially  candidateRegenerated  \
 0                   5                           5                     0   
 
    candidateRecovered  reservePlansActivated  candidateRecoveryReplans  \
 0                   0                      0                         0   
 
    finalCandidatePortfolio  
 0                        5  ,
 'attempts':   candidateId  schemaValid  hardLockPreserved  semanticAnchorPreserved  \
 0          C1         True               True                     True   
 1          C2         True               True                     True   
 2          C3         True               True                     True   
 3          C4         True               True                     True   
 4          C5         True               True                     True   
 
    planFidelity anchorDecision fidelityDecision  contentLanguageValid  \
 0          True           PASS          ADAPTED                  True   
 1        

,A,B,판정,Family A,Family B,겹침,실질 차이,단계,semantic judge,관계 설명
0,C1,C2,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
1,C1,C3,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, valuePropositionThes...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
2,C1,C4,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, dataDependency, valu...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
3,C1,C5,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, dataDependency, valu...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
4,C2,C3,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, valuePropositionThes...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...
5,C2,C4,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, dataDependency, valu...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
6,C2,C5,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","customerInteractionModel, dataDependency, valu...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
7,C3,C4,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","dataDependency, valuePropositionThesis, offerT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
8,C3,C5,DISTINCT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","dataDependency, valuePropositionThesis, offerT...",PRIMARY_BUSINESS_CHOICE,False,핵심 solution 또는 주요 Business Architecture 선택이 다릅니다.
9,C4,C5,VARIANT,INTERMEDIARY:OTHER,INTERMEDIARY:OTHER,"businessRole, operatingModel, partnerModel, de...","valuePropositionThesis, offerThesis, solutionT...",MEANINGFUL_THESIS_VARIANT,False,Architecture family는 유사하지만 target/use case/val...


## 26. Legal Fact Completeness + Business Design Completion + C1 Fact Pattern


In [27]:
prechecks = [engine.legal_precheck(item) for item in candidates] if RUN_STAGED_LEGAL else []
display(show_legal_precheck(prechecks))
legal_preparation = (await engine.prepare_legal_candidates(seed, candidates)
                     if RUN_STAGED_LEGAL and candidates else None)
candidates_before_legal = candidates
candidates = legal_preparation.candidates if legal_preparation else candidates
display({'factCompleteness': [item.model_dump(mode='json') for item in legal_preparation.reports]
                              if legal_preparation else [],
         'roleSemanticBatchCalls': legal_preparation.roleSemanticBatchCalls if legal_preparation else 0,
         'dependencySemanticBatchCalls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
         'completionAttempted': legal_preparation.completionAttempted if legal_preparation else 0,
         'completionValidated': legal_preparation.completionValidated if legal_preparation else 0,
         'completionAccepted': legal_preparation.completionAccepted if legal_preparation else 0,
         'completionExhausted': legal_preparation.completionExhausted if legal_preparation else 0,
         'completionCompliance': [item.model_dump(mode='json') for item in legal_preparation.completionCompliance] if legal_preparation else [],
         'preLegalExclusions': legal_preparation.excludedCandidates if legal_preparation else []})
display(show_legal_fact_pattern(candidates[0].candidate, seed) if candidates else [])

,candidateId,label,directSeller,intermediary,regulatedPhysicalActivity,personalDataDependency,qualificationDependency,riskHints
0,C1,Structural risk precheck — not final legal review,False,True,True,True,False,"[물리 활동, 개인정보]"
1,C2,Structural risk precheck — not final legal review,True,True,True,True,False,"[물리 활동, 개인정보]"
2,C3,Structural risk precheck — not final legal review,True,True,False,True,False,[개인정보]
3,C4,Structural risk precheck — not final legal review,True,False,True,False,False,[물리 활동]
4,C5,Structural risk precheck — not final legal review,False,False,True,False,False,[물리 활동]


{'factCompleteness': [{'candidateId': 'C1',
   'status': 'COMPLETABLE',
   'missingDesignFacts': ['sellerRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'contradictions': [],
   'completionRequirements': ['sellerRole의 역할 존재·부재와 해당 책임을 의미에 맞게 명시해야 합니다.'],
   'affectedFields': ['sellerRole'],
   'roleSemantics': [{'field': 'platformRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'MATCH',
     'semanticUsed': False,
     'semanticStatus': 'NOT_RUN',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'providerRole',
     'status': 'MATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'MATCH',
     'semanticUsed': False,
     'semanticStatus': 'NOT_RUN',
     'finalStatus': 'MATCH',
     'safeReason': '역할 의미가 필드와 일치합니다.'},
    {'field': 'sellerRole',
     'status': 'MISMATCH',
     'presence': 'PRESENT',
     'deterministicStatus': 'AMBIGUOUS',
     'semanticUsed': True,
     'semanticStatus': 'MISMATCH',
     

,field,legalFact,source,authority,decision
0,platformRole,여행 계획 지원 플랫폼으로서 사용자와 제공업체를 연결한다.,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
1,providerRole,여행 계획 서비스를 제공하는 주체로서 사용자와의 상호작용을 담당한다.,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
2,sellerRole,숙박 및 체험 제공업체로서 예약 서비스를 판매한다.,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
3,intermediaryRole,플랫폼을 통해 사용자와 제공업체 간의 거래를 중개한다.,CONCEPT_GENERATED,REVIEWABLE,PROPOSED
4,transactionFlow,"[사용자가 앱에서 여행 일정을 선택한다., 사용자가 예약을 진행한다., 예약이 완료...",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
5,paymentFlow,"[사용자가 예약 시 결제 정보를 입력한다., 결제가 완료되면 예약 확인이 이루어진다...",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
6,personalDataUsage,[사용자의 이동 가능 시간과 관심사를 수집하여 맞춤형 여행 일정을 제공한다.],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
7,physicalActivities,"[배송, 현장 방문, 설치]",CONCEPT_GENERATED,REVIEWABLE,PROPOSED
8,partnerRequirements,[숙박 및 체험 제공업체와의 제휴 계약 체결이 필요하다.],CONCEPT_GENERATED,REVIEWABLE,PROPOSED
9,qualificationRequirements,[],CONCEPT_GENERATED,REVIEWABLE,PROPOSED


## 27. Prepared Legal C1 Evidence Summary


In [28]:
legal_adapter = CurrentLegalAdapter() if RUN_STAGED_LEGAL else None
legal_c1_input = (legal_adapter.task_input(candidates[0].candidate, seed)
                  if legal_adapter and candidates else None)
display({'candidateId': candidates[0].candidateId if candidates else None,
         'externalFacts': legal_c1_input['externalFactContext']['facts'] if legal_c1_input else [],
         'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'})

{'candidateId': 'C2',
 'externalFacts': [],
 'note': '공식 근거 수와 allowed index는 Full Legal 응답/실패 diagnostics에서 확인'}

## 28. Full Evidence Judgment — C1 Staged Smoke


In [29]:
RUN_FULL_LEGAL_C1 = RUN_STAGED_LEGAL
legal_one = None
if RUN_FULL_LEGAL_C1 and candidates:
    try:
        legal_one = await engine.review_legal_candidate(seed, candidates[0])
        display(show_legal_result([legal_one]))
    except Exception:
        display(show_legal_failure(candidates[0].candidateId, engine.gateway))
else:
    print('SKIPPED — RUN_FULL_LEGAL_C1=True로 명시해야 실행됩니다.')

,candidateId,route,productionStatus,sourceStatus,evidenceCoverage,reviewPhase,factCompletenessStatus,legalSourceStatus,finalEvidenceJudgmentExecuted,recoveryResolution,...,legalClarificationCount,safeSummary,unknownFacts,requiredControls,requiredPartnersAndQualifications,requiredDisclosures,prohibitedVariants,evidenceCount,evidenceRefs,evidenceDiagnostics
0,C2,ACCEPT,IMPLEMENTABLE_WITH_CONTROLS,SOURCE_COMPLETE,None,LEGAL_SOURCE,None,SOURCE_COMPLETE,True,None,...,0,사용자 데이터 보호 및 개인정보 처리 관련 법령 준수 필요.,[],"[통신판매업자는 신고를 해야 한다., 통신판매업자는 거래조건에 대한 정보를 제공해야...",[숙박 및 체험 제공업체와의 제휴 계약 체결이 필요하다.],[사용자의 이동 가능 시간과 관심사를 수집하여 맞춤형 여행 일정을 제공한다.],[],24,"[{'referenceIndex': 0, 'sourceType': 'OFFICIAL...","{'coverageStatus': 'SOURCE_COMPLETE', 'coverag..."


## 29. C1 Route + Staged Redesign/Compliance/Second Legal


In [30]:
c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = ([], [], [], 0, 0)
if legal_one and candidates:
    c1_portfolio, c1_legal_all, c1_required_inputs, c1_redesigned, c1_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[:1], [legal_one])
display({'initialRoute': legal_one.route.value if legal_one else 'SKIPPED',
         'redesignRequirements': legal_one.redesignRequirements if legal_one else [],
         'recoveryReviews': [item.model_dump(mode='json') for item in c1_legal_all[1:]],
         'requiredInputs': c1_required_inputs, 'terminalCandidates': len(c1_portfolio),
         'redesigned': c1_redesigned, 'replanned': c1_replanned})

{'initialRoute': 'ACCEPT',
 'redesignRequirements': [],
 'recoveryReviews': [],
 'requiredInputs': [],
 'terminalCandidates': 1,
 'redesigned': 0,
 'replanned': 0}

## 30. Remaining 4 Legal + Exhaustive Recovery Summary


In [31]:
RUN_REMAINING_LEGAL = RUN_STAGED_FULL
legal_remaining = []
portfolio, legal_all, required_inputs = (list(c1_portfolio), list(c1_legal_all), list(c1_required_inputs))
redesigned_count, replanned_count = c1_redesigned, c1_replanned
c1_terminal = bool(c1_portfolio or c1_required_inputs or (c1_legal_all and c1_legal_all[-1].route.value == 'SYSTEM_FAILURE'))
if RUN_REMAINING_LEGAL and c1_terminal and len(candidates) > 1:
    legal_remaining = await engine.review_legal(seed, candidates[1:])
    rest_portfolio, rest_legal, rest_inputs, rest_redesigned, rest_replanned = await engine.resolve_legal(
        seed, candidate_preparation.usedPlans, candidates[1:], legal_remaining)
    portfolio += rest_portfolio; legal_all += rest_legal; required_inputs += rest_inputs
    redesigned_count += rest_redesigned; replanned_count += rest_replanned
else:
    print('SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.')
legal_initial = ([legal_one] if legal_one else []) + legal_remaining
display({'recoveryTrace': [item.model_dump(mode='json') for item in legal_all
                           if item.candidateId not in {x.candidateId for x in legal_initial}],
         'requiredInputs': required_inputs, 'metrics': engine._legal_metrics})
print({'Plan Selected': len(selected_plans),
       'Candidate Generated': candidate_preparation.candidateGenerated if candidate_preparation else 0,
       'Candidate Valid Initially': candidate_preparation.candidateAcceptedInitially if candidate_preparation else 0,
       'Candidate Regenerated': candidate_preparation.candidateRegenerated if candidate_preparation else 0,
       'Candidate Recovered': candidate_preparation.candidateRecovered if candidate_preparation else 0,
       'Fact Completion Attempted': legal_preparation.completionAttempted if legal_preparation else 0,
       'Fact Completion Validated': legal_preparation.completionValidated if legal_preparation else 0,
       'Fact Completion Accepted': legal_preparation.completionAccepted if legal_preparation else 0,
       'Dependency Semantic Calls': legal_preparation.dependencySemanticBatchCalls if legal_preparation else 0,
       'Completion Compliance PASS': sum(item.status == 'PASS' for item in legal_preparation.completionCompliance) if legal_preparation else 0,
       'Legal Ready': len(candidates) if legal_preparation else 0,
       'Legal Reviewed': len(legal_initial),
       'Legal Accepted': sum(item.route.value == 'ACCEPT' for item in legal_all),
       'Legal Redesigned': redesigned_count, 'Legal Replanned': replanned_count,
       'Final Portfolio': len(portfolio)})

SKIPPED — C1이 정상 terminal에 도달한 후 remaining Legal을 실행합니다.


{'recoveryTrace': [],
 'requiredInputs': [],
 'metrics': {'factAttempted': 4,
  'factValidated': 1,
  'factAccepted': 0,
  'factExhausted': 4,
  'dependencySemanticCalls': 1,
  'completionCompliancePassed': 0,
  'providerNoncompliant': 1,
  'recheckFailed': 0,
  'redesignAttempted': 0,
  'redesignValidated': 0,
  'redesignAccepted': 0,
  'redesignExhausted': 0,
  'replanAttempted': 0,
  'replanValidated': 0,
  'replanAccepted': 0,
  'replanExhausted': 0}}

{'Plan Selected': 5, 'Candidate Generated': 5, 'Candidate Valid Initially': 5, 'Candidate Regenerated': 0, 'Candidate Recovered': 0, 'Fact Completion Attempted': 4, 'Fact Completion Validated': 1, 'Fact Completion Accepted': 0, 'Dependency Semantic Calls': 1, 'Completion Compliance PASS': 0, 'Legal Ready': 1, 'Legal Reviewed': 1, 'Legal Accepted': 1, 'Legal Redesigned': 0, 'Legal Replanned': 0, 'Final Portfolio': 1}


## 31. Replan


In [32]:
display(show_replan(type('PortfolioView', (), {'concepts': portfolio})()))
print({'replanned': replanned_count, 'reserveAvailable': len(reserve_plans)})

""


{'replanned': 0, 'reserveAvailable': 1}


## 32. Final Portfolio


In [33]:
legal_terminal_status = ('READY_FULL' if len(portfolio) == MAX_CONCEPTS else
    'READY_LIMITED' if portfolio else
    'LEGAL_RECOVERY_COMPLETE_NO_ACCEPTED_CANDIDATE' if legal_initial and len(legal_initial) == len(candidates)
    else 'LEGAL_PENDING')
display(show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else {'status': legal_terminal_status})

,candidateId,lineageId,parentCandidateId,이름,핵심 작동방식,family,descriptor,수익,운영
0,C2,L2,None,주말 여행 계획 지원 서비스,자동으로 생성된 일정은 사용자가 쉽게 예약할 수 있도록 관련 링크를 제공한다.,거래 중개 · 기타 운영,{'thesis': {'targetSegmentThesis': '주말 1~2일 단기...,"수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다.",사용자 데이터를 기반으로 알고리즘을 지속적으로 개선하여 더 나은 여행 계획을 지원한다.


## 33. Unresolved Candidate Summary


In [34]:
display(show_required_inputs(required_inputs) if required_inputs else {'unresolved': []})

{'unresolved': []}

## 34. Manual Concept Selection


In [35]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 수정
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
print({'selectedCandidateId': SELECTED_CANDIDATE_ID})

{'selectedCandidateId': 'C2'}


## 35. 7 Hypotheses


In [36]:
hypotheses = (engine.build_or_load_current_hypothesis_contract(selected_concept)
              if RUN_STAGED_FULL and selected_concept else [])
hypotheses = await engine.resolve_hypothesis_semantics(hypotheses) if hypotheses else []
display(show_hypotheses(hypotheses))
display(show_hypothesis_readiness(hypotheses))

,HypothesisType,ProposedValue,FinalValue,SemanticStatus,SemanticReason,Locked,DecisionStatus,LegalImpact
0,TARGET_REGION,한국 전역,None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
1,REVENUE_MODEL,"수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다.",None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
2,PRICE,서비스 이용 시 발생하는 예약 수수료는 5%로 설정한다.,None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
3,CHANNELS,"모바일 앱을 통해 사용자와 소통하며, 소셜 미디어를 통한 마케팅을 진행한다.",None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
4,DIFFERENTIATORS,"사용자의 이동 가능 시간과 관심사를 실시간으로 반영한 지원, 숙박 및 체험 예약 페...",None,VALID,실제 사업 가설값입니다.,False,PROPOSED,NONE
5,PRE_MARKET_SOM_SHARE,targetSharePercent=15.0 horizonYears=3 rationa...,None,VALID,수치·기간·산식 가설이 명시되었습니다.,False,PROPOSED,NONE
6,PRE_MARKET_SOM,amount=5000000.0 currency='KRW' period='연간' ca...,None,VALID,수치·기간·산식 가설이 명시되었습니다.,False,PROPOSED,NONE


{'All Values Semantically Valid': True,
 'All Decisions Confirmed': False,
 'Ready For Handoff': False,
 'status': 'NOT_READY',
 'reason': 'UNRESOLVED_HYPOTHESES',
 'unresolvedHypotheses': ['TARGET_REGION',
  'REVENUE_MODEL',
  'PRICE',
  'CHANNELS',
  'DIFFERENTIATORS',
  'PRE_MARKET_SOM_SHARE',
  'PRE_MARKET_SOM']}

## 36. Confirm / Edit


In [37]:
CONFIRM_ALL_PROPOSED = True
HYPOTHESIS_EDITS = {
    # 'PRICE': '월 17,900원',
}
confirmed_hypotheses = engine.confirm_hypotheses(
    hypotheses, HYPOTHESIS_EDITS, confirm_all_proposed=CONFIRM_ALL_PROPOSED) if hypotheses else []
display(show_hypotheses(confirmed_hypotheses))
hypothesis_readiness = show_hypothesis_readiness(confirmed_hypotheses)
display(hypothesis_readiness)

,HypothesisType,ProposedValue,FinalValue,SemanticStatus,SemanticReason,Locked,DecisionStatus,LegalImpact
0,TARGET_REGION,한국 전역,한국 전역,VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
1,REVENUE_MODEL,"수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다.","수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다.",VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
2,PRICE,서비스 이용 시 발생하는 예약 수수료는 5%로 설정한다.,서비스 이용 시 발생하는 예약 수수료는 5%로 설정한다.,VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
3,CHANNELS,"모바일 앱을 통해 사용자와 소통하며, 소셜 미디어를 통한 마케팅을 진행한다.","모바일 앱을 통해 사용자와 소통하며, 소셜 미디어를 통한 마케팅을 진행한다.",VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
4,DIFFERENTIATORS,"사용자의 이동 가능 시간과 관심사를 실시간으로 반영한 지원, 숙박 및 체험 예약 페...","사용자의 이동 가능 시간과 관심사를 실시간으로 반영한 지원, 숙박 및 체험 예약 페...",VALID,실제 사업 가설값입니다.,False,ACCEPTED,NONE
5,PRE_MARKET_SOM_SHARE,targetSharePercent=15.0 horizonYears=3 rationa...,targetSharePercent=15.0 horizonYears=3 rationa...,VALID,수치·기간·산식 가설이 명시되었습니다.,False,ACCEPTED,NONE
6,PRE_MARKET_SOM,amount=5000000.0 currency='KRW' period='연간' ca...,amount=5000000.0 currency='KRW' period='연간' ca...,VALID,수치·기간·산식 가설이 명시되었습니다.,False,ACCEPTED,NONE


{'All Values Semantically Valid': True,
 'All Decisions Confirmed': True,
 'Ready For Handoff': True,
 'status': 'READY',
 'reason': None,
 'unresolvedHypotheses': []}

## 37. Actual Delta Legal


In [38]:
RUN_DELTA_LEGAL = True
delta_legal_result = None
if RUN_STAGED_FULL and RUN_DELTA_LEGAL and selected_concept and any(h.deltaLegalRequired for h in confirmed_hypotheses):
    delta_legal_result = await engine.review_delta_legal(seed, selected_concept, confirmed_hypotheses)
    confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, delta_legal_result)
display(delta_legal_result.model_dump(mode='json') if delta_legal_result else {'status': 'NOT_REQUIRED_OR_SKIPPED'})

{'status': 'NOT_REQUIRED_OR_SKIPPED'}

## 38. Market Seed


In [39]:
handoff = None
if selected_concept and legal_all and hypothesis_readiness['Ready For Handoff']:
    handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
display(handoff.marketAnalysisSeedSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'market-analysis-seed-snapshot-v1',
 'schemaVersion': '2.0',
 'snapshotId': 'lab-market-seed',
 'projectId': 0,
 'selectionId': 0,
 'conceptId': 'C2',
 'createdAt': '2026-08-10T06:51:42.104530+00:00',
 'sourceSnapshotHash': 'sha256:c80d3ca735a837897aa27872d5da439460d4fc5e7040a0ac3a7c3f46aea70473',
 'originalSeed': {'ideaOverview': '사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동 구성하고 숙박·체험 예약 페이지로 연결하는 여행 계획 서비스',
  'fields': {'ideaOverview': {'value': '사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동 구성하고 숙박·체험 예약 페이지로 연결하는 여행 계획 서비스',
    'source': 'USER_INPUT',
    'decisionState': 'LOCKED'},
   'problem': {'value': '짧은 주말 여행은 이동 시간까지 고려해 일정을 직접 조합하는 데 많은 시간이 든다.',
    'source': 'USER_INPUT',
    'decisionState': 'LOCKED'},
   'targetUsers': {'value': '주말 1~2일 단기 여행을 자주 계획하는 직장인',
    'source': 'USER_INPUT',
    'decisionState': 'LOCKED'}}},
 'aiInterpretation': {'interpretedProblem': '짧은 주말 여행을 계획하는 데 소요되는 시간과 노력을 줄이는 것.',
  'interpretedTargetUsers': '주말에 1~2일 단기 여행을 자주 계획하는 직장인들.'

## 39. Marketing Source


In [40]:
display(handoff.marketingSourceSnapshot if handoff else {
    'status': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'contract': 'marketing-source-snapshot-v1',
 'schemaVersion': '2.0',
 'snapshotId': 'lab-marketing-source',
 'projectId': 0,
 'selectionId': 0,
 'conceptId': 'C2',
 'marketAnalysisSeedSnapshotId': 'lab-market-seed',
 'marketAnalysisSeedSnapshotHash': 'sha256:8b86432c2ef173e0402afc31488201cea0a8afc80e76e260f94b94c2670f38c6',
 'createdAt': '2026-08-10T06:51:42.104530+00:00',
 'conceptName': '주말 여행 계획 지원 서비스',
 'targetSegment': '주말 1~2일 단기 여행을 자주 계획하는 직장인',
 'problem': '짧은 주말 여행은 이동 시간까지 고려해 일정을 직접 조합하는 데 많은 시간이 든다.',
 'valueProposition': '사용자는 여행 계획을 쉽게 진행할 수 있다.',
 'positioning': '사용자의 이동 가능 시간과 관심사를 바탕으로 주말 근거리 여행 일정을 자동 구성하고 숙박·체험 예약 페이지로 연결하는 여행 계획 서비스',
 'keyFeatures': ['사용자의 이동 가능 시간 입력 기능', '관심사 기반 여행 일정 추천', '숙박 및 체험 예약 연동 기능'],
 'targetRegion': '한국 전역',
 'revenueModel': '수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다.',
 'price': '서비스 이용 시 발생하는 예약 수수료는 5%로 설정한다.',
 'pricing': '수수료 기반 모델로, 사용자가 예약 시 발생하는 수수료로 수익을 창출한다. · 서비스 이용 시 발생하는 예약 수수료는 5%로 설정한다.',
 'channels': ['모바일 앱을 통해 사용자와 

## 40. Contract Compatibility


In [41]:
display(show_downstream_handoff(handoff) if handoff else {
    'contract': 'NOT_READY', 'reason': hypothesis_readiness.get('reason'),
    'unresolvedHypotheses': hypothesis_readiness.get('unresolvedHypotheses', [])})

{'호환성': 'PASS',
 '구조': 'STRUCTURE_PASS',
 '계약': 'CONTRACT_PASS',
 '필드 매핑':                        v2Field                             downstreamField  \
 0        candidate.conceptName        selectedConcept.identity.conceptName   
 1  candidate.solutionMechanism  selectedConcept.solution.solutionMechanism   
 2                hypotheses[*]                             finalHypotheses   
 3                        legal                                 legalResult   
 
               source  transformed  required  
 0  CONCEPT_GENERATED        False      True  
 1  CONCEPT_GENERATED        False      True  
 2     USER_CONFIRMED         True      True  
 3  OFFICIAL_EVIDENCE         True      True  ,
 'Market payload': {'contract': 'market-analysis-seed-snapshot-v1',
  'schemaVersion': '2.0',
  'snapshotId': 'lab-market-seed',
  'projectId': 0,
  'selectionId': 0,
  'conceptId': 'C2',
  'createdAt': '2026-08-10T06:51:42.104530+00:00',
  'sourceSnapshotHash': 'sha256:c80d3ca735a837897aa278

## 41. Trace


In [42]:
display(show_trace(engine.trace))

,순서,시각,stage,action,entity,parent,status,mode,호출,요약,decision,reasonCode
0,1,2026-08-10T06:48:00.070939+00:00,CREATED,CREATED,NaN,NaN,RUNNING,LIVE,NaN,V2 Lab 실행을 생성했습니다.,NaN,None
1,2,2026-08-10T06:48:09.269618+00:00,SAFETY_CHECKING,IDEA_BRIEF_DERIVED,NaN,NaN,PASS,LIVE,1.0,Idea interpretation/readiness를 보존했습니다: READY_F...,NaN,None
2,3,2026-08-10T06:48:09.269632+00:00,SAFETY_CHECKING,READINESS_INCONSISTENT,NaN,NaN,WARNING,LIVE,NaN,READY_FOR_REVIEW이지만 score=0입니다. V2 gating은 막지 ...,READINESS_INCONSISTENT,None
3,4,2026-08-10T06:48:09.301614+00:00,SEED_ANALYZING,ANALYZED,lab-idea-brief,NaN,PASS,LIVE,NaN,필수 3개와 LOCK 3개를 분류했습니다.,NaN,None
4,5,2026-08-10T06:48:09.301864+00:00,SEED_ANALYZING,DESIGN_SPACE_READY,NaN,NaN,PASS,LIVE,NaN,Open=11 Constrained=0 Breadth=EXPLORE,NaN,None
5,6,2026-08-10T06:48:09.349657+00:00,PLANNING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,최대 5개 동적 plan을 요청합니다.,NaN,None
6,7,2026-08-10T06:48:43.269025+00:00,PLANNING,DRAFTS_GENERATED,NaN,NaN,PASS,LIVE,2.0,Plan draft pool=6,NaN,None
7,8,2026-08-10T06:48:43.270326+00:00,PLANNING,NORMALIZED,NaN,NaN,PASS,LIVE,NaN,System metadata를 부여한 Plan=6,NaN,None
8,9,2026-08-10T06:48:53.958564+00:00,PLANNING,ARCHITECTURE_SEMANTIC_FALLBACK,NaN,NaN,PASS,LIVE,3.0,low-confidence Plan architecture 6개를 batch 분류했...,NaN,None
9,10,2026-08-10T06:48:53.958609+00:00,PLAN_VALIDATING,STARTED,NaN,NaN,RUNNING,LIVE,NaN,Opportunity·LOCK·DUPLICATE/VARIANT/DISTINCT 관계...,NaN,None


## 42. Provider/Legal Usage


In [43]:
display(show_provider_usage(engine.gateway.usage))
print('상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.')

,논리 작업,상위 외부 작업,논리 stage별,상위 외부 작업 stage별,재시도,소요(ms),모드별,token,보고 비용
0,23,23,"{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...","{'SAFETY_CHECKING': 1, 'PLANNING': 1, 'NORMALI...",0,220883,{'LIVE': 23},None,None


상위 외부 작업 수는 내부 AI/MOLEG 네트워크 호출 수와 동일하다고 주장하지 않습니다.


## 43. Replay Manifest


In [44]:
display(show_replay_manifest(engine.gateway))

{'status': 'REPLAY_PARTIAL',
 'entries':                    operation  \
 0               LEGAL_REVIEW   
 1          SEMANTIC_RELATION   
 2    NORMALIZE_ARCHITECTURES   
 3                     EXPAND   
 4                  PLAN_POOL   
 ..                       ...   
 425                   EXPAND   
 426    LEGAL_FACT_COMPLETION   
 427    IDEA_BRIEF_DERIVATION   
 428                PLAN_POOL   
 429             LEGAL_REVIEW   
 
                                                   hash operationVersion  \
 0    00489dd62bff3c4b9e8711093fb3e398a39dde1547de32...               v3   
 1    00c1eeb4f93989a6db3891fb7a22355a71a68800a6f641...               v3   
 2    013ebfc11d61a317c1041ddcf75902b1dab0d310e73666...               v1   
 3    02bf209b45f806d67ec78f5bdc5e0e24bf99e05fbd297f...               v3   
 4    038e1d9753b8c07ad835823495b54906a14fb6542c3c70...               v3   
 ..                                                 ...              ...   
 425  fc18f5423064548add8b9929

## 44. One-click MOCK


In [45]:
mock_result = None
if LIVE_TEST_LEVEL == 'FULL_E2E' and MODE != 'LIVE':
    mock_result = await ConceptPortfolioEngine('MOCK').run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=True)
display(show_run_summary(mock_result) if mock_result else {'status': 'SKIPPED'})
assert mock_result is None or (mock_result.handoff and mock_result.handoff.contractStatus == 'CONTRACT_PASS')

{'status': 'SKIPPED'}

## 45. One-click REPLAY


In [46]:
RUN_ONE_CLICK_REPLAY = False
replay_result = None
if RUN_ONE_CLICK_REPLAY:
    replay_gateway = ProviderGateway('REPLAY', recordings_dir=RECORDINGS_DIR)
    replay_result = await ConceptPortfolioEngine('REPLAY', gateway=replay_gateway).run_full(
        TEST_INPUT, max_concepts=MAX_CONCEPTS, auto_confirm_hypotheses=False)
display(show_run_summary(replay_result) if replay_result else {'status': 'SKIPPED'})

{'status': 'SKIPPED'}

## 46. One-click LIVE


In [47]:
RUN_ONE_CLICK_LIVE = True
live_result = None
if RUN_ONE_CLICK_LIVE and LIVE_TEST_LEVEL == 'ONE_CLICK':
    assert MODE == 'LIVE', 'MODE=LIVE를 먼저 명시하세요.'
    one_click_gateway = ProviderGateway('LIVE', recordings_dir=RECORDINGS_DIR)
    one_click_engine = ConceptPortfolioEngine('LIVE', gateway=one_click_gateway)
    live_result = await one_click_engine.run_full(TEST_INPUT, max_concepts=MAX_CONCEPTS,
                                                  auto_confirm_hypotheses=False)
display(show_run_summary(live_result) if live_result else {'status': 'SKIPPED'})
if live_result:
    display(show_live_validation_summary(LIVE_SCENARIO, live_result))
    display(show_required_inputs(live_result))
    display(show_pre_legal_exclusions(live_result))
    display(show_legal_resolutions(live_result))
    if live_result.runStatus.value == 'FAILED':
        display(show_run_failure(live_result))
        display(show_provider_failure(one_click_engine.gateway))
        display(show_provider_usage(live_result.providerUsage))
        display(show_trace(live_result.trace[-20:]))
        display({'unresolvedCandidates': live_result.unresolvedCandidates,
                 'lastSuccessfulStage': live_result.failureDiagnostics.lastSuccessfulStage if live_result.failureDiagnostics else None,
                 'firstFailedStage': live_result.failureDiagnostics.firstFailedStage if live_result.failureDiagnostics else None})

{'status': 'SKIPPED'}